# Pneumonia Chest X-ray Classification

## 1. Problem Statement

To develop a machine learning model that can automatically analyze and classify chest X-ray images into predefined categories (such as normal and diseased conditions) in order to assist in accurate, fast, and reliable diagnosis of chest-related diseases.

## 2. Domain Analysis

This project belongs to the healthcare and artificial intelligence domain, focusing on medical image analysis. It involves using machine learning, especially deep learning models, to analyze chest X-ray images and detect diseases. The aim is to assist doctors by providing faster and more accurate diagnosis, improving healthcare efficiency and accessibility.

## 3. Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

## 4. Loading Dataset

In [2]:
train_dir = r'C:\Users\Dinesh H L\Datascience\Machine Learning\Pnemonia\chest_xray\train'
categories = ['NORMAL', 'PNEUMONIA']
filepaths = []
labels = []

In [3]:
for category in categories:
    folder = os.path.join(train_dir, category)
    for fname in os.listdir(folder):
        filepaths.append(f"{category}/{fname}")
        labels.append(category)

In [4]:
df = pd.DataFrame({'Filename': filepaths, 'Label' : labels})

## 5. Data Preprocessing

In [5]:
datagen = ImageDataGenerator(rescale = 1./255, validation_split = 0.2)

In [6]:
train_generator = datagen.flow_from_dataframe(
    dataframe = df,
    directory = train_dir,
    x_col = 'Filename',
    y_col = 'Label',
    subset = 'training',
    batch_size = 32,
    seed = 42,
    shuffle = True,
    class_mode = 'binary',
    target_size = (150,150)
)

Found 4173 validated image filenames belonging to 2 classes.


C:\Users\Dinesh H L\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\legacy\preprocessing\image.py:918: UserWarning: Found 2 invalid image filename(s) in x_col="Filename". These filename(s) will be ignored.
  warnings.warn(


In [7]:
valid_generator = datagen.flow_from_dataframe(
    dataframe = df,
    directory = train_dir,
    x_col = 'Filename',
    y_col = 'Label',
    subset = 'validation',
    batch_size = 32,
    seed = 42,
    shuffle = True,
    class_mode = 'binary',
    target_size = (150,150)
)

Found 1043 validated image filenames belonging to 2 classes.


C:\Users\Dinesh H L\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\legacy\preprocessing\image.py:918: UserWarning: Found 2 invalid image filename(s) in x_col="Filename". These filename(s) will be ignored.
  warnings.warn(


## 6. Model Creation

In [8]:
model = Sequential([
    Conv2D(32, (3,3), activation = 'relu', input_shape = (150, 150, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation = 'relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation = 'relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dropout(0.5),
    Dense(128, activation = 'relu'),
    Dense(1, activation = 'sigmoid')
])

C:\Users\Dinesh H L\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(optimizer = Adam(), loss = 'binary_crossentropy', metrics = ['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       4,735,104 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,828,481 (18.42 MB)

 Trainable params: 4,828,481 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Model Fit

In [23]:
history = model.fit(
    train_generator,
    steps_per_epoch = train_generator.samples // train_generator.batch_size,
    epochs = 20,
    validation_data = valid_generator,
    validation_steps = valid_generator.samples // valid_generator.batch_size
)

Epoch 1/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 64s 490ms/step - accuracy: 0.9739 - loss: 0.0773 - val_accuracy: 0.7383 - val_loss: 1.0318
Epoch 2/20
  1/130 ━━━━━━━━━━━━━━━━━━━━ 41s 318ms/step - accuracy: 1.0000 - loss: 0.0429

C:\Users\Dinesh H L\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


130/130 ━━━━━━━━━━━━━━━━━━━━ 13s 102ms/step - accuracy: 1.0000 - loss: 0.0429 - val_accuracy: 0.7441 - val_loss: 1.0156
Epoch 3/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 63s 483ms/step - accuracy: 0.9754 - loss: 0.0728 - val_accuracy: 0.8076 - val_loss: 0.9069
Epoch 4/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 1.0000 - loss: 0.1275 - val_accuracy: 0.8154 - val_loss: 0.8963
Epoch 5/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 64s 493ms/step - accuracy: 0.9833 - loss: 0.0650 - val_accuracy: 0.8467 - val_loss: 0.8084
Epoch 6/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 14s 110ms/step - accuracy: 1.0000 - loss: 0.0549 - val_accuracy: 0.8301 - val_loss: 0.8410
Epoch 7/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 66s 510ms/step - accuracy: 0.9848 - loss: 0.0566 - val_accuracy: 0.7969 - val_loss: 0.9271
Epoch 8/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 14s 103ms/step - accuracy: 1.0000 - loss: 0.0212 - val_accuracy: 0.7656 - val_loss: 1.0091
Epoch 9/20
130/130 ━━━━━━━━━━━━━━━━━━━━ 65s 496ms/step - accuracy: 0.9865 - loss: 0.0511 - val

## 8. Prediction Test Data

In [17]:
test_dir = r'C:\Users\Dinesh H L\Datascience\Machine Learning\Pnemonia\chest_xray\train'
test_filepaths = []
test_labels = []

In [18]:
for category in categories:
    folder = os.path.join(test_dir, category)
    for fname in os.listdir(folder):
        test_filepaths.append(f"{category}/{fname}")
        test_labels.append(category)

In [19]:
test_df = pd.DataFrame({'Filename': test_filepaths, 'Label' : test_labels})

In [20]:
test_datagen = ImageDataGenerator(rescale = 1./255)

In [21]:
test_generator = test_datagen.flow_from_dataframe(
    dataframe = test_df,
    directory = test_dir,
    x_col = 'Filename',
    y_col = 'Label',
    batch_size = 32,
    shuffle = False,
    class_mode = 'binary',
    target_size = (150,150)
)

Found 5216 validated image filenames belonging to 2 classes.


C:\Users\Dinesh H L\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\legacy\preprocessing\image.py:918: UserWarning: Found 2 invalid image filename(s) in x_col="Filename". These filename(s) will be ignored.
  warnings.warn(


## 9. Model Evaluation

In [24]:
pred_probs = model.predict(test_generator)
preds = (pred_probs > 0.5).astype(int).flatten()
true_labels = test_generator.classes
class_labels = list(test_generator.class_indices.keys())
print(classification_report(true_labels, preds, target_names = class_labels))
print(confusion_matrix(true_labels, preds))

163/163 ━━━━━━━━━━━━━━━━━━━━ 40s 243ms/step
              precision    recall  f1-score   support

      NORMAL       1.00      0.80      0.88      1341
   PNEUMONIA       0.93      1.00      0.97      3875

    accuracy                           0.95      5216
   macro avg       0.96      0.90      0.93      5216
weighted avg       0.95      0.95      0.94      5216

[[1068  273]
 [   5 3870]]


## 10. Final Conclusion: Pneumonia Chest X-Ray Classification using CNN

## 11. Future Improvements